In [4]:
import os
import sys
import yaml
import chromadb
from llama_index.core import VectorStoreIndex, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama


# 0. Path Fix (Because we are in the notebooks/ folder)

# This tells Python to look in the parent directory (the root) for config.yaml
sys.path.append(os.path.abspath('..'))
config_path = os.path.join('..', 'config.yaml')


# 1. Load Configuration
with open(config_path, "r") as file:
    config = yaml.safe_load(file)


# 2. Initialize Models
print("Loading Embedding Model and Local LLM...")
Settings.embed_model = HuggingFaceEmbedding(model_name=config['embedding']['model_name'])

# We add num_ctx to restrict Ollama's RAM usage to fit comfortably on your laptop
Settings.llm = Ollama(
    model=config['llm']['model_name'], 
    request_timeout=120.0,
    context_window=4096,
    additional_kwargs={"num_ctx": 4096} 
)

# 3. Connect to Database
print("Connecting to local ChromaDB...")
# We use '..' to point back to the root directory where chroma_db is saved
db_path = os.path.join('..', config['data']['persist_dir'])
db = chromadb.PersistentClient(path=db_path)
chroma_collection = db.get_or_create_collection(config['data']['collection_name'])
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

index = VectorStoreIndex.from_vector_store(vector_store=vector_store)


# 4. Ask the AI a Question
# CHANGE THIS QUESTION based on the PDF you actually downloaded!
question = "What are the specific requirements mentioned in the document?" 

print(f"\n Question: {question}")
print(" AI is thinking... (This might take a minute on CPU)")

query_engine = index.as_query_engine(similarity_top_k=3)
response = query_engine.query(question)

print(f"\n Answer:\n{response.response}")


# 5. Verify Citations
print("\n Sources Cited:")
for node in response.source_nodes:
    domain = node.metadata.get('domain', 'Unknown')
    file_name = node.metadata.get('file_name', 'Unknown')
    score = node.score 
    print(f"- Domain: [{domain.upper()}], File: {file_name} (Relevance Score: {score:.3f})")

Loading Embedding Model and Local LLM...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 786.76it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting to local ChromaDB...

 Question: What are the specific requirements mentioned in the document?
 AI is thinking... (This might take a minute on CPU)

 Answer:
Some specific requirements mentioned in the document include:

- Flood Damage-Resistant Materials Requirements
- Free-of-Obstruction Requirements
- Corrosion Protection for Metal Connectors and Fasteners in Coastal Areas
- Requirements for Flood Openings in Foundation Walls and Walls of Enclosures

 Sources Cited:
- Domain: [STRUCTURAL], File: fema_nfip-technical-bulletin-9-09292021.pdf (Relevance Score: 0.509)
- Domain: [STRUCTURAL], File: fema_nfip-technical-bulletin-9-09292021.pdf (Relevance Score: 0.489)
- Domain: [STRUCTURAL], File: fema_nfip-technical-bulletin-9-09292021.pdf (Relevance Score: 0.487)
